# Micro Insurance Claim Risk & Anomaly Detection — EDA & ModellingThis notebook covers the full workflow: data cleaning, EDA, feature engineering, a supervised fraud classifier, an anomaly detector, evaluation, and example reason-code + human-review output.**Dataset:** Insurance Fraud Detection (auto claims, 1000 rows).  **Target:** `fraud_reported` (Y/N).> On Kaggle: attach the dataset, then set `DATA_PATH` to the path shown by the file-listing cell.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, RocCurveDisplay)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Load data

In [ ]:
# On Kaggle, uncomment to find your file:
# import os
# for d,_,fs in os.walk('/kaggle/input'):
#     for f in fs: print(os.path.join(d,f))

DATA_PATH = 'insurance_claims.csv'   # adjust to your path
if DATA_PATH.lower().endswith(('.xlsx','.xls')):
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 2. Clean dataThe `?` string is this dataset's missing-value token. We convert it to NaN and fill categoricals with `unknown`.

In [ ]:
MISSING = ['?','','NA','nan','None']
df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].astype(str).str.strip().replace(MISSING, np.nan)
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].fillna('unknown')
for c in df.select_dtypes(include=np.number).columns:
    df[c] = df[c].fillna(df[c].median())
print('Missing after clean:', int(df.isnull().sum().sum()))
df['fraud_reported'].value_counts()

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x='fraud_reported', data=df, hue='fraud_reported',
              palette=['#4c9f70','#d1495b'], legend=False)
plt.title('Class balance: fraud vs legit'); plt.show()
print('Fraud rate: {:.1f}%'.format((df.fraud_reported=="Y").mean()*100))

In [ ]:
t = df.assign(f=(df.fraud_reported=='Y').astype(int)).groupby('incident_severity')['f'].mean().sort_values()
t.plot(kind='barh', color='#d1495b', figsize=(6,4)); plt.title('Fraud rate by incident severity'); plt.xlabel('fraud rate'); plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.kdeplot(data=df, x='total_claim_amount', hue='fraud_reported',
            fill=True, common_norm=False, palette=['#4c9f70','#d1495b'])
plt.title('Total claim amount by class'); plt.show()

In [ ]:
num=['total_claim_amount','injury_claim','property_claim','vehicle_claim','policy_annual_premium','months_as_customer']
plt.figure(figsize=(6,5)); sns.heatmap(df[num].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Numeric feature correlation'); plt.show()

## 4. Feature engineeringInterpretable ratio/flag features that are useful fraud signals.

In [ ]:
df['claim_to_premium_ratio'] = df['total_claim_amount'] / df['policy_annual_premium'].replace(0,np.nan)
df['claim_to_premium_ratio'] = df['claim_to_premium_ratio'].fillna(0)
df['injury_share'] = (df['injury_claim'] / df['total_claim_amount'].replace(0,np.nan)).fillna(0)
df['is_new_customer'] = (df['months_as_customer'] < 12).astype(int)
df['no_witnesses'] = (df['witnesses']==0).astype(int)
df['no_police_report'] = (df['police_report_available']!='YES').astype(int)

DROP = ['policy_number','insured_zip','incident_location','incident_date','policy_bind_date',
        'auto_model','policy_csl','insured_hobbies','auto_make','incident_city','insured_occupation']
y = (df['fraud_reported'].str.upper()=='Y').astype(int)
X = df.drop(columns=[c for c in DROP if c in df.columns]+['fraud_reported'])
print(X.shape)

## 5. Build & train the supervised classifier`class_weight='balanced'` handles the 25/75 imbalance.

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()
pre = ColumnTransformer([('num', StandardScaler(), num_cols),
                         ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.25,stratify=y,random_state=42)
clf = Pipeline([('pre',pre),('m',RandomForestClassifier(
        n_estimators=300,min_samples_leaf=2,class_weight='balanced',random_state=42,n_jobs=-1))])
clf.fit(Xtr,ytr)
proba = clf.predict_proba(Xte)[:,1]; pred=(proba>=0.5).astype(int)
print(classification_report(yte,pred,target_names=['legit','fraud']))
print('ROC-AUC:', round(roc_auc_score(yte,proba),3))

In [ ]:
cm = confusion_matrix(yte,pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['legit','fraud'], yticklabels=['legit','fraud'])
plt.xlabel('predicted'); plt.ylabel('true'); plt.title('Confusion matrix'); plt.show()
RocCurveDisplay.from_predictions(yte, proba); plt.title('ROC curve'); plt.show()

## 6. Feature importance (global reason context)

In [ ]:
ohe = clf.named_steps['pre'].named_transformers_['cat']
names = num_cols + ohe.get_feature_names_out(cat_cols).tolist()
imp = pd.Series(clf.named_steps['m'].feature_importances_, index=names).sort_values(ascending=False).head(12)
imp.sort_values().plot(kind='barh', figsize=(6,5), color='#3d5a80'); plt.title('Top fraud signals'); plt.show()

## 7. Anomaly detection (Isolation Forest)

In [ ]:
iso = Pipeline([('pre',pre),('m',IsolationForest(
        n_estimators=200, contamination=float(y.mean()), random_state=42))])
iso.fit(Xtr)
flags = (iso.predict(Xte)==-1)
print('Flagged anomalies in test set:', int(flags.sum()))

## 8. Example decision-support outputCombine fraud probability + anomaly flag into a recommended action.

In [ ]:
def recommend(p, anom):
    if p>=0.60 or (anom and p>=0.30): return 'HIGH-PRIORITY REVIEW'
    if p>=0.30 or anom: return 'MANUAL REVIEW'
    return 'AUTO-APPROVE (LOW RISK)'

res = Xte.copy()
res['fraud_probability']=proba.round(3)
res['is_anomaly']=flags
res['recommendation']=[recommend(p,a) for p,a in zip(res['fraud_probability'],res['is_anomaly'])]
res[['fraud_probability','is_anomaly','recommendation']].sort_values('fraud_probability',ascending=False).head(10)

## 9. Summary & limitations- Random Forest reaches ROC-AUC ~0.77 with balanced recall on the minority fraud class.- Isolation Forest adds an unsupervised safety net for unusual claims.- Output is **decision-support**: a human officer makes the final call.- **Limitations:** small dataset (1000 rows), auto-insurance only, historical labels may carry bias, and the model must not be used to auto-reject claims without human review.